# CinderTrace — Sri Lanka 2022 and 2024 data checkpoint

The 2023 checkpoint was independently verified locally: 219 raw NASA responses, 13,366 matched observations and 48 matching monthly totals. Its ZIP SHA256 is `509afdf0418a01b248671addb4d7a221a02e0e5e275fb203aa6e170206d29077`.

This notebook extends **observed data**, without changing the frontend or generating harmonized values. It queries the mirror for the same polygon in 2022 and 2024, then independently retrieves NASA standard-product rows for advertised archive dates. It retains native confidence, quality, published type and satellite provenance. No observations are gap-filled or silently discarded.

## Run in order

1. Open in Colab or VS Code. Install dependencies; restart the kernel if the DuckDB version check asks you to.
2. Use the same free FIRMS MAP_KEY as before, entered only at the hidden prompt. No Earthdata password or additional upload is needed: the verified boundary is embedded.
3. Keep the same results folder when resuming. Each completed extraction/request is checksummed and cached. Expect roughly 500 small NASA requests across both years, depending on advertised NOAA-21 dates. A request failure stops the run, never becomes zero, and retains verified progress.
4. Save this executed notebook and the final ZIP for local verification. A discrepancy is a review item, not an automatic change to the original data.

**2022 scope:** Terra, Aqua, S-NPP and NOAA-20. **2024 scope:** those four plus NOAA-21. NASA availability is queried before planning dates. An unavailable standard stream or pre-archive dates stay explicitly unverified, with null authoritative counts. NRT mirror rows remain in the raw extract and separate diagnostics; they are never compared as standard rows or combined into a calibrated headline.

The 2024 daily grid includes February 29. Whole-year observing coverage remains unverified even when every supported query matches. Source notices describe global gaps; local impact needs observing evidence beyond hotspot detections.


In [37]:
%pip -q install "duckdb>=1.5,<1.6" pandas pyarrow numpy shapely


## 1. Set up durable output
A fresh run folder is created the first time. Rerunning this notebook reuses its active folder. Change `NEW_RUN` to `True` only when deliberately starting a new snapshot. No Colab upload calls are used in VS Code.


In [38]:
from pathlib import Path
from datetime import datetime,timezone
from io import BytesIO
from urllib.request import Request,urlopen
from urllib.error import HTTPError,URLError
from getpass import getpass
import base64,zlib,hashlib,json,sys,time,uuid,zipfile
import importlib.metadata as metadata
import pandas as pd
import numpy as np
import shapely
import duckdb
from IPython.display import display
assert tuple(map(int,duckdb.__version__.split('.')[:2]))==(1,5), 'Restart the kernel after installing DuckDB 1.5.x, then run from this cell.'
IN_COLAB='google.colab' in sys.modules
NEW_RUN=False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/cindertrace_atlas/verification/lk_2022_2024')
else:
    BASE=Path.cwd()/'atlas_lk_2022_2024'
BASE.mkdir(parents=True,exist_ok=True)
active=BASE/'active_run.txt'
if NEW_RUN or not active.exists():
    run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
    active.write_text(run_id)
RUN_ID=active.read_text().strip()
assert RUN_ID and '/' not in RUN_ID and '..' not in RUN_ID
OUT=BASE/RUN_ID;OUT.mkdir(exist_ok=True)
CACHE=OUT/'cache';CACHE.mkdir(exist_ok=True)
print('Run:',RUN_ID)
print('Results:',OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run: 20261114T201123Z_30cc8ecb
Results: /content/drive/MyDrive/cindertrace_atlas/verification/lk_2022_2024/20261114T201123Z_30cc8ecb


## 2. Verify the reference geometry
This is the exact Sri Lanka pilot polygon, not an administrative substitute.


In [39]:
BOUNDARY_SHA256='838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e'
BOUNDARY_BYTES=zlib.decompress(base64.b64decode('eJyFfUnLtseV3l8xWpuPmk9Vtg1ZJRDIMnghupVGxG0ZRb0wxv8911VVZ6jntfJJBttvnbvGM0/P33/47W9//emH//KHH/7rTz/+9p+//vQvv/z5zz/9628///KXH/74hx/+9/nj/wXA//r7JygB/vrrL3/96dffft4gf//h53/DfyX8/S8//scG/Z+//vyH//bjX/7Pjz/8A3/9959++Y+ffvv1bxtWZ/vv//nn337+H7/8+W//ftb8119++fXffv7Lj7+dZfGPrG8ibeT+xz/Mb2WsPtr40x//cP5eS8tj7X9kj8tKIjY+eyurn/Fxvpc5+kr7n2ZgtY71TDPKqsvGvyz/J47src2c2+r5TkiYWmUWKXfRdeeYqZXZ6wHA7PMuN6YC5DLHrOev68DVJHmcqWuxicaYsx2wueFKa2nlu/1q87WW61mwpDxwM+cTWzDNVVKKE2XBOW04rZRzuLva0pB2764a3HcvwC9LpmBPPSzZsT5e6XydxS58zTI3QFu1dtuGAczSxkzhxdpMXbr+ZRrcSLmPsLmGGx1JF9TLElxfnhFR2tiPc+BKt/kWLqXH+UYd2fBtcep6hjfCNCBg7rr97G9Yi556P1Frc81678xOuVYvo4braphd6n18CcvOKfkAABnKPAB9GUBJ5VxTX1WK3OM7AUifussDN4Ykvbji9LY6bj5e0xQpza4H/0fGedc8McWF9NfoLcXb45oz1y+v26WusgH4+vPeaCm+3+/gUcA5cAQgkL3J+pbxgnUqHuuTjIIZ7xVssCSldyO7VUYGSd2N7PFcCjjJ3fsIcKNNI2PAgTaBxR/kCXQrRu5y5kvkDB9XvnIF81nPumO1qU83hRi3/w7MGO0Ts8FnZMwcv2+VN/7JVkapeLG47yapGvvCZc/a9t/xOMZDes357H7gjpfej+1ugIXVai/OWVstQ59ODA5PqssC5bNyD6MYcARc0HNdo4Dt6TUBmVvfqEe8L236e/Q0I2qDtfRqfO+BKy2Q5JIOLLb3z9hWlTiP9CZGq3OBqXdF+D0O9LdtzAXKLCWgPuBLXpcS7RVwYbP3wEAWtpWV/4nd60qjP/xv9d6X4mfzV824pnyupZFYPylpNUhQFQ3n/nGQpnBGupCHpUlED2CBMZAxIRtyj9PIbEU+8YFSti0XFoADbuZxlzOhMiaouRx0xanAdu5E1SYCw6uK7IeqifW6oG4bqN56PePgBFXJ1Rj/97iDMxIgLbDZZcT61sEgl7IhQ9UhYAz1jNc+h/154WV7ODp4YSnDr6bgyiKhg7YgcKfxNH2JKq0984Cfgx1euOlHH7k44i+IUogiUWGYfT5OEJ6uycDSyohMhqTWZy5h3QahYeoCODvQ/xw7UR1QZLDzgQy7is5zvjZakytNuhjc9645Kl/4d8Yzgn9S6nySQSbtRCY4ypbAd07jclCUDjZRbeD9n3l0GKwYVB+PsKBd6GPYVc06Zsln7wtKlr0dBdRSJjvPeMej3zfJNkEDV53y+xtpeTV7oQPWR21fyL8DsMsD10AHPg11yXh9YCvNVUgIYQjB+HmmcJtKHDoNFVo9wwFbDVgVRMs4RIj5IZBMdYTSQRkTnhoLLuXChsrgYuCKe1xAq2HdVGe8hT7Ak1VbaK5tgRTKiHBC3v1FA4fm1uv7OvhC9PLt9hNwToXawXhwfyiC9y8GB/It+Z0PInYoBdp0UF5UHq375hVqmk/T0vtKEG9gf/cLA4P6kmbYFc48m3Ic01G/TzNBdwK6QzE+QOAdKt+rK+q9UVfzI0JLgnWibNbtNEj45yZmIy7dvdl0wOGpNwr+ruq3cWvqVuNZDgiVDQdMn/39fUdjpEiWhzimQHVWWRS0VqgJqpBsuAFTYxkDFYrWEi+091YVbUzDEuJu7w9cwlXpy9re8bj9QYeOW2lf+LaAM+TnHQUCURHFJdx3jxn5aSP+BljJpAG9PsdsGEs5UsCAJmhyNFCeQMCpZNpwa2vqbqN+Z727t5m+FdjH581n7WAc+3n4915lSA33NSHRoNNEJkI4IH3rexN4PNy76t5yAWBul1XuJmElKlLpBLDzGgTvQQDIwnTvfvYLQMYuRzIW8lHF7mrjNY0UEQB0AmFpwx28RsJlNfCcbtrzBYOohlEYwcCymhQdBrrLcIsXw7iMqarb0NMKOFM/GhawJi9TwRyg0oQLb4xrB+dZkZ9w21B3RouaOYQX5KhqMnZ6bDyNMw6hONQm1usVPE6SZ+OQRKk9spFwkKegtrivDBU2qWai1wm2CdCokkKFvZohh1sLh9nDUA6x/sc9QUeAUIqsK8Oe6rpctflkQj2I24KIEODRHYeAqHm5sKIdsqozYp0HCAG16oyDyS919Ng4/Rm9hWtKDdtRvCmKJvTtgIGer/u15+pIPg7lsKjdiJcJHPscB/gRj32EqN7nMrgBQnGzCHAgyG6ce9i2IV3btOumPQL1Qufza8p8c/cwbDMb1qB+qa8LJCi9jLguUGAomk8DaxX25DHDIVrUDB+2r0xfWvSmLU5dFTttHoHZqqrqAYOoNN6xoEjBijpmIVQadbaJ7xfo3a5deRU0/hkcAZrc+TPMW5Vk2Q8A2irXnqzAT984LKwcfUOYF1JTtTWbAHZZG9eUnBSGNoCTqmJ5dtChDXZFbJ+gg3U+livesYYXy9xYHKceojfdbR5gowSZAThMAh3OTopLE7kDQKHlVzcpb8NJC70eqiXajVRoyqXFjbzzVPA81Zp0nukvBAMdp4/DUAcl2f4bmLNaIeelgeVVLSwHA9n16NrD+/alXNzY2AKVlB69r7g2yCXVGuyZBj69ALCwhvoIpwHgI+gMETVhvxXlyCadFpXBNeLGJqYeyhBIFCC1550zjH7VH88BM+io0Fv6vGOHjqESKRucgPL0PXs5tj0HcBVzYzR51HFp8M8wJ1d3ix7Dsl/zkROAg0kc7UjAdfKMx6dJuJlKzgcAn0Ai36MZAOinK0CBya6P5jP0ebyZE2LbnBlVh3nf6tLZF4FFoJso6SschGHveYSDyeJ21SDR5TLNmeP13ArNfJzeGwCGZYqUBjipakHbTWItKmjGGydlK/0sdz59CAgyPP8ep8te59GLxrbzTBFhaF3V1/tMuAJV5/GMQWEthi+42CIPotPFNdUNfMmTcAPiet/PAH1n5bZ6ehAa1D4VUwcsQWd79bwDN8528fy12t9h7+fHZYZN1KXoClRrvccIQJdRZNj2GhUBFWvn8wJVS13TBgeZnkqPy8h2w9g84LG5xHFMa+p9tVttDQjwwDXY2XYbhW7ViHzgJ1eJ2GdN4E7nK3pp5pfLzrPm5zTQbZM5EGye2o/s6AP6ksbNbJlCK9993TwrnshIkNQ9Dsr3gxP3euM2skStgB4lGDb+lis9ISrYjdnEnl0Grj+bJ/4sh3ltFrysccj7tB1y4bE+NxzmOcNL+SKoBOy1j2ePQv+0Km5GTZCmZY3nMsBfVKOtDgeJWZ4gyDii++NM2Ex7pgPvz2piFZ8OJ5PxTDdgZ+pxjXlARdkKsCMMyKypwOsBbpq/+Vz5fjtVBZ07wNIv+vWBA8YaoWefD2z8mY/GsjmYa3jDYrzlwOGWTKHz80IjH2ciOkkMlXDksaL0B0Fg375fcJRx3g/LFLU/Zg0AN+aIS8Flq4izFTKsjLOzSp1HdXl/sU9+SZlRplEDDNjUosKFyyrJ8QzX1KKeRYepPQv+57zYSe6hTq5aDAAor2ZRv8ckxRmzXwxengGIO3Na2voCND/6FxSToYobxYmA0SifOxPMudT8HbYD4Fw556YpV5+oHsfxl5ZjJKbnzZA/N4KLhsYT4SiJ2xMZ23AQzzWHEzdalWrJOMbACsrLnRAMWjLG0j8xH3RU78VL8WWgJhwFnN6WKvPzM2jiFx8h52cgTGiCEh+EamxzdKM5HW8V6FuaPIGOzSLpvnlCxYNHtHFG81p75gEqmaAzOOjCs9Rwq2CKwGcfBjXfGHPLTrEuHxlcPOMMELk8I5L1u7Eqys2bCQkIDvCU+2UOO4Ld1VtUOI9+qi8vPgHIbUU4+pfVoeuCHypMWZGDQGKBvpR1m2Sl97a5A4UPA5bZo499g4GjXGygc/3+GUIXBPhQeVrQipSuFYw26w1xYxNdydWGIb6nEhtVXx1gwF+iMgWkpgPy41YZe5PnUqrQTam7B2uEtIhSpjJWoVLGuB4ouZ/UFlyJO45MhDMSdnNIoOs01Y9sle7BiQOWGQXV0yocwxm9Rd0ekgO6tU6np2/4syt+Gw7qvqg/0IgHkgEM0H2a8xvNeOelUEvAv+ITg7vRtWjjkGdyPsTjygof5pQilYCrpTU+FB+gNG4jGt+QHTiNeiEVDOjdH6dHzjUb5hqraBDh49AQg+jVPKK2HtNKjlKLcdyvIki2laB4Pcot+Wr1cSlMe9oDWN/+vKghH/hMlVDVa304RiYmNUChfSjAOR3IEDchArXHYYqoF1F3TuzoK0e4ycCb4oxeVeeNJvc3cT5a0vlzPoj0VB2vOd+sxutPeomrekKvSB1G2w4Hcyck6gjtdpmmpEFEgqmNuAyVL305Q8ReGA9zNxThGCRUxPbrgs6hLtaz3r5u1W5sPuoI+Ww8M5/jSRvZAAJ6dXFAuCbGAzE21v07PWTT9wnidkyUne/Qm40DUUsucdqyZes9R7Prh57RSnwmnLc+8Z4NhqP289ygtfrhfc7UZbcScUTp3c8y7chMcKA1Xk8BYK8phthKk4aQcwzCOZfEZRRo6OeaZmBIWH9Ic/tHvk1oHBYcz3bdgyFUd/oI8208ryEbvkCkpBbRd9LkttimEhz9CCunOB9lsu534PmZTufoSxXbDguxN6dqRmcVCLNmiVC2CgN/4xwL7NR88dnW4WOPOBHk9nKxptc/yPJTDWgDO6pN86or2BRq4oFahAFhFQMmTCCgsMkzDySipRuZPShUEHokEwhmPJSqSfosQpeTeK6E0GSTbuaNPgsuF2ats3MBg6eWpedUMIhiaDcB+6GJq8sdw6NPM7z2MDMuW9IDZ4MDIT9wBcRazJvdDA70MHO4/dLpyfnwL03qZ3UjSWYOoTknfF8QzuqZ3bcFZYce5XsvijYTbztSXA+SpYz1ZF7cc1Zg3QBP7dBolBh8PQFTcuIHnNAD8PmKQH1InrwBoBvnkT8nYqw8uCLHN0azu0ZHDP0gKcoO4wKg0/2jN2QTVQYa1Kuw4Zjzqml8ZqxOZtqISyvAVcgF1crMA0Bqa00iHLhwM1+avfRW+hcBZHJJFdg+EXEshxNCL14mzmz/oDjPV9hg4K4BfzFjC7HNQU8dxKmdizRRzoeJiZl6g0ZQoMFy3hTSe/TxaRxBK+IURiGD+TetqxPTWArUJ15YePvRNvrdLxWMtkl1BgawROfpk2244RiB9ujb+AbVHzLZxin7Zb8AbCZg/5PRSgBozec7ELS7erA4pPS+EIhleqx0YGRw/B4WbJRjRpvGymCUjDPOxAYTkANSJcuZuIL/WZxWqZXBYXMy7YtsUK2SeQmchacRrHyAZSqOKihUnlBWdg7XnQM83muGod5ktjAL7AGQvhrXJtaZnFw9J2TQSigynlyxDUfRciait8bWc7UDrLs9C3YqGHoLtjHgSKs9LtiY9mb6JkO6Ei4JuuBytZ4BibniKoV+Id2ugVF4uU4JMBiyphUa32sweEfwPI9veHCaTRfOloWEkpD3A7jOJPsn857WjtAQiNyGpulUvmJyAHpbsRyGvT/GFpa65IwrVW5mrQgHwKwhBVMuaiVWbmyAGtdH/fQ5g/3sHD5HbjrBy5ubvCeCUGtnvBXD7co85/MZuFHW8xgXAocBc9mvSWeoiX8zWumozH0dgM7U7I+FsTcIhDPDhLFrHMK9E/3ovYOOqpNAuI/oUQR686PsAdlkK1+wlfAc5CQBAcE7gPIWjIFC1w9jyY0KcPp4FOhxtUtEGog8xsU+FwJqzf0mULNhZLiXi0k+kQIYv87ujSL5dQ9jYBxUazqmu5XSKOXhpBmakN0HraoRKl+4DMP7iosWw1qsd3CNasNB+ClPsOPg9ms+EzHffvn3VGvP38kAVDD4PhsD/vG8UGLFbFybh9bjjPvNTBtUajIw3ESKpJ13lYqqPhYEBBpa/OKuWsGGLDbXt056vrq7ZzKeXUtiRum5j0QvpEWhGcd8eEFirNninQrWGHkaYZ+Jl2reOwvOgnRH8PIRDvj18doHbu5XTkKj3NMreCy3QzHeGdlQS9+yVCB3LQa3j4VnwRE1WKjBehBfkpDuQLjeLAfNsgsm6NVqBDZrSBAM1QoxFE5ojYT8P8JBfzNXj50DLLaGoO2gCir6XmnX1cy08QJSqS5L9gGrL+93eMZqeo6eS+hbxeedaSVgZ7a/trOO7f44npjPc7+3pCoiXRsRDpaWJ5jbfEDZltcDB5ZiOVPAgRb8Tp0acsoWxLAkLXDMuc44azn871U/m7yg+8qe2wXpyeSgznSibo6q4d8zo8OvqzM7gCbXhdNcACiLacwV9zloq3wmiYEO+k552xMVN02LAUh1AXgmgjSy7yfkwhjPfpiFp0qRPh/lm4Q8ZcJBwbFpoL634TFVDEPwLmMemntBbSeWAxIOMrsrn7BUtMKqKjd/CQdbQIsDLLlpZCZKDmOenXTKNMJLZ7Yug3X1wR7IVMv760zcaCVOAwQ02mXZxkpnu0CtrJVzosjZmSgVrJcNBymidu0yOIZjDwCljLIx2ydIBAShvouzT6bTKnZTMq4QKeG9YG+2jwZ5RCnOv0PPtlQ7sMcxxpmPB33tvcQaPphy98NEtfiDHhp0/P6SKcMVVuWoG4RKDqujXAwYy2o2lwHU2UIdyj5BdgpqzMtd6RkfPVs1lNg8jLm442Bj0rQ4es6+3r6iAMe6AkuT0xtq1B0vIUHBnuszP45OWTCxA5CqJTnbOEbrcr1mU8gwF6blxYHjjxZy07khchXVMnXjlTh39kNHY9ODl0Gzx0P6m6F2Mc6guMDgjYRwEhkzs3e6jVcY4kdypE5d6YNWC3Oa2+X4ZCI6l81QWWsYNZzEvHXLFytAmRbCfxQpsMvNCV5snrlt0CBCgItDE8E8j7gsqql2c0eUTUUWQzFwWGhQ0RJOzPscnyjEjKKR1jMfNtJtX5mOhajCkIe0bN+nxeDB+RA8Qr/LfK+R4r0wz0pHcY8gwf3nsXVmHSjMmT4PsklKZUIzAOBEu5rm5vR3a3pBiQQbEsSoGWemeNyLXAbHONxRumDRddU+i75smniO52VZkrsso8IWZEXdOFrgokGmKDINoNL8P5YLH+V1fwCgZ/D6xzDiG5iEVjCm0jQJT0rvWJL8uV4VGdcQY7WW0sk0gAItcB2LlLm9w//ekgXM9zgwtyj9mKBj9dE5T93hEHUg2fw4WhsRl9tOx1PjxIoVF8vxrndhWlnRIh8eOZyzFfArK3bTuhyInk67+7zrhuu7fup+2Qyutm4pqxuO1aj65bRa18n02ePtgAQS+SxFZwJLLdGyZiy9lSfquSfKvYorAoMx+b6soGkxHel5SBBUnf3JDdzlk7Ax8z4Zo4LNNGcrBcIfLHVnn4xRSFEmZUV7rG9N54p2RbqJAi9xgr1cSzgZVPrh9bMtYfHoV+ET5fRY+LswrVfL8T5wEHPdjmr7KYW+obBvVn3W6sMyN94w1tGm1WlgSS/D3uN0ShnTttMURrfanQCbfEj7ALSZ8rNPkqjiiFfIrVRydJFgn90iaV4XTDzOz/1Al/JqHCtqaqQC1+j2upCDqql7yRj1H4/n8BzgEV7gtzhT4MnQ/sTj2cWnoZ8hbr8zpXF8WW4wuh+31Zle9QWN2o57n/enWejnAsrJvSCmekVH4Bn39+7newGnzI5+qT7u3UXf/Hg0w1PHXSxKeg++HcOG57DOVuQcTCde68lWPtXoTSeAKdTlMWHOSWEOOcsQugtaU2nQHZHw6sMNC2EejniykHefWNADPJQgTDrEH2Ii46Ef6d2TcQgGglXL0qvlIfKtamIcuMJWBoGAZP+d+ZPDXKt2flx2FQnbIUeQ/AT0N9wke7f7JBwJ7kmIOHBz5hhvhcyFCaKOFOcjvFOnO4aMAvdgvkKK0U4K+Kn34S0/oCBI3nEyekXEKzg7jLp5zp2GWOGF3wtQdrz73AWpT97phuuDFe1hn9ioFWdWn4+p1Z79uOFgtKn9afhNb/Y5EZSdUp/0gz3ei5gYMXRrX+EoBM8BoTLYhQhQM5QecBTiJPReSLtekX8XupA/jzt679O1ccDtsJtei188DUK3x4V+VGbj3pWdemYq80GvtW5Sxqk7bqbYiU1j9m9gc3TrBXxgvdpS7PMOMQXvlSNWgbhmsexlpxoYNBLDrFBtZgrMuZsT68ZFGVLzbh6MxsVki06vff9smSGsAy41bJv566VHbW63NBAWEYflgBm4du+WQq1//70wRGQF80yqXGdecknVzZuvzxTpHo4pOxlDbTBbgE4nZZYn+YDkY2zBFgQHKMWTASbdlizKufPZ/SQJStVJ7gEOW/8YA8s0s04WEcNY6+NecMU3yQj4b6di34Eqp7vR7jfgy5aVdIANZKzLS8J5qvszOE7UG+9+mELa3mSuTGVVqd/2lSszh8PxWKbT+xNhO8vi25h5BzjqszpeWE93Bph42+rH+Wmm7ZyvAkbrpS5WNAx9QU69Bz/PFkXwKuac2tvpiIWI3nFByCtiCUEdvBcvAB7QGuK9EcnknzQZGni/vdMKBUjMu+5tdticKvYqocwJTa9wnzEjlLHurEqMo7OweO8cA6ZKi5FKDrP4ZT27BQcxp7B3EIE8GE9+fGUFlDlZbNOsyZWY/su7S/3JQ95w0MZPjiC0fe9MIb6vYU0JzmtDlR2al2siSfhST23B9m1MX2eU93wYL5aS5cuBM8q7XHEPj7WOYOcsyz1eB44pimrieWk6lNLgYOGy07Olndl1lqu4igk4eqDa/NheZcTbBdxk7LKs+an4MtuNKTOByHDY9Dq493ywsE6eOU7JrEmfIOdDHJVV3dmfv57s3gJT2zpmhQ4BfJrbVC03MzYdfRjQOBvKzMqz74CzJda/FNZlzifF59wnFNHLDgZz4T4uiDIixNs4D4wpK1B1dO+r9BkviCU4Fr+3i2SidI+pn4UuW0sGcm4CJnl6cmXWtEjgMqvFMkLYmTjnJ9rRLdk9oDPpIxHzF/kqfIuY5ps7pWnM0z1YwmSHw/vp2PjoJ8JUvxz0D4JBU1EW6ZcEwX0q5lgrsfLjWN/zMF0rNkFiMDBZIMP3U2BNxxT/xEfUbQVspFK1981zLlWIulNn79G8nvQJkUDvxo062eMoxeqQxCIHExGGzY3Z3CVcBDBtNB+vYHXFnXbMm6XQeXLR7/OtsU4eK8NVX26q03vqbIB5rEBlWweSs8j5O5j9NAPaFF+QqVUZ30TXwXyHjxeDwTKkRlVvybZpP/gDtDr8xzMzhH5jL8OwC59AdAu8HDjqrkbV3mUEsi/FDENWBWdfDSLibGew2kjNWrez6CyNKiO9njkFfwjVqbgNcjMV2cFvAnagTO6oYrTTl69D5uB+lZMv7LZlbTkaaWsXaukibqTRU/IkX7PNj9VvGhgLxx4czM1j/MHGBE5f3MPy3o+xsAzm4HgG+JNWuPXXmi0FvR849vNRXuBqLDB8ujOStMWEL1efl6aKC7DCkpWM4y7yWfcxbeKF+Pv0Dk4WsFlm8IVrlmUcGtIw9hbLWjMR2FpcuEsAJtKIIoxMyLOD3LrpTJePcHXrjjbemMR8mCXt3dj9yfpIHC43KCueivi94ZliVizhSCiKxO1354MSDYvVDk76uAonVppvR5JjkWRVzUtm7vW9JBtnvDgaEK/MZm5kaRExIfvA/hUH/aUa5eYBAN1qCNXdEDu/yO3OowRYAnVgW7TaYm02K0jdS5Hp7Xz2S/+G/sURAps5Xw/22flCRzIsN//ot4xFWva6+1hA3Le2Ba9opY5+PRWY3Ny6Z+dS6qifhDV5pBqFAuNzosV2RhiTTD8/indnMOSLs42pEO1URDHoMoI3j6lce4CC3b0csJDSU9nU2WTg08k1WOS5h4Ve0idZ47grVr6lgdRC9T58HFbdLQZO7hxzpwz74D4tTDDPsGJRlxEM0+WoBUCl97CrYdXWH8qtHWWBoxKQ92RkgfZTSw3+li2hW8F2e6xYzAwtJFmEz/p7rUZNK2IndAWqXvdlHW6r2iY0WNCem2VxWFdVSL76ViGxx1+TJx1/w+EN9jzQnlhqb39fcsiAOZfN12cFSYhaTSYPd9NAFx/mKf2aTEHrNl5YHBetw8lE1vVh30FaNKvWO10QsHnX+Xw6gS1y+jik4n1JyWVPH5sE0rNQffbP8mpxehByk6/9UmvRviiZRRlPasg+rexs6zBPYWutz9Zhi3hYIq+jN6BZva3vK10nydr1/rqf5ddyq2CpMhf1FBqRr12i6QlihJPiOZkKNhgG8QIXdl9hv5PP3nSLfuDH7FjMjW6fmvdOW8rP8UBwxToAWKQRnDB1d36x743QHrfxTlXvNtbp3eLxiRVAc2pn1a6ml6X6QP6NHOLehFt52cRt9hIygdgIaDGJ6R7EQpwky3I7hNLPEol0x3RZBt7iMtCvsxXbVZsHl9luw042lPgyPoo8Tf/oBHQTOES1oYI6sQOur2nVLNnC47vtdmw9x9YFloI5s8H1kZ+2WdRuWxo2zvSK3W+K0tkSPKgpgBbOB1It49BS6Cjf2nC1bbGgvMrbynpPNKAUuHq2aOkv61PjmQLMehk1bBRMBPxcZb8lDiRIzKfLIh0bVoZkaUyZBkswjdhvi3anBdknSTk2pKoM3HRDIGHXjxTO9zE+6G17+p5hG17RoGB84hLvHwo207k/js/02jJPWzP6gfPHcRLrVWvs+lfZVSe/FWqEY4p9icdi5aBlgurtsP1TimjdIIPFusHZK49L4Gu34HblwQCgrMnpDkfH6XqclHuc+c/xuUCYpHnbiKRS4zOxXVfL4ZlW6acZJ72TzS+WeBnbHHambAx/nzza7RHarNA43DflxW2aCsybhl47b/f0fGRnStWrHZ9ZXdPiuqyqUvewJfUxACTzdvdkxtz9ws5dqNKFd2K/5DmiJ2Tnz6QO5em0p63DUzsyzeaQybvb9678Wik7PWd3iYnbZcjZWLfD4RRPJ+ReGdZWdULfo7DRRop8FWzWayctNwWKO3MWzkTsrWkTkE3N2wCYnQ3Uh2IJTXQHjBpWaEtKsYfH/2K/jNPZl6lU8l4ZZKSXMej3LOZ8CZNW7agHc3extkpwS8AS5lfcxsK4AHXd2gSgvznkdig2QgUm1NNuGQSRPWnPtwfz0j5bydIrPUGMTWXcR7B2OuLyfTF2eje29U+/WZDVPTE5lTzRyQ1ABfMcubM81KZkp6MWpXRjTrg1EV0OB3y6EzAGb2l+aSfjnoHO1ssXbaxZZWKUJnLTxvVKdB9vMBDq05SZotc7eSsYW0yopXSuCLrjsHT4fzJNF/9xCXuLuttSRaHcdlf9t6CErTiBdKE+dZ+ylo9mS4Tb/XQjHJC5WcLr5hznRZmUrn9lQ4aHlBtdq5YORs9VCeFyfk4ZauN0l1RPJiBCZ6/qMs5FQZAuak6wAA3YWdIn2+bV2yearb/y4+XbAGlYOs2hbWhp3nDa0loTS5djF3c6j9t8ijAIxl/qiAdjVaWYe9auDSbEbevf6d02GV4nb+pwR4Ziv6BTp887dmWEYVezud+bYycbuMf9TubrvJVUZGpsSBhJc7AsdTyW6WbOzTsO3p65JKhPHTazzcc4PV4hzKelkhnAjvY8vXxZzKoTmTAYTBeMfWsH8wrNTHFpwuyDEvfFAgKLupjMbmVekQouWR6P3U4F5dSxNSqMezK7j12RgtSdeB5pUg1+63O5K/b8iLvaoQ+jN5OYc44WNQXQKujEDAImk6fIxJhDIOaucU185hwCf7xTEKNty+Am21rHVvyD3RwsIRZ6KLA3/mIG80GmOa1dsZ1lFHfbct+V2dM6Xln7Fn86AE8HDmfnAoNaUt5zL0e6YMkwQvTsB7zMpgGp9P6iEg2L8rHdxKK86V6EjXJ8x/sXt9wYdc9xW419dM0w5e9VzDMBblv8z9MTLvfonGlav1izQ+es5XY/ztMKycz7utjzc9VnnsZQc+AA2+6llHo6Q+9S9xIv79jHNZ9mzEl2x/+D3LbcYAglShxou1D4zW5mgH+4T46xqcQ2kDbO/jqeyLVjTtUzexYLo1sKx5HdvPHJuD37YN/bZx+jVas+HQ5Xamg8ss/FeOLdoft82mihcnc/Ry9mndl7tEKiicTAZpn9KSzYcEzVdWcP4ZrLanE3C9PC7/vuWmgbSM28JPVe9LJaGPfFMUHw6OnCLPDgH2KDj3auuOPTjwAvWAvTAM84+26puLYJ2K2y1Hh1nd0vlXkbmLA/7h4fOM74CPQuFmA/uoWwBsY9KLtVy/k77ZPPH1Fi9p3c+UEgYi2XDKXAC68eyKItS3w2/+jatkM8CElBvlSx4Av8u29se+utHMpIvQxPJd6PP3eEWNHdWEdr7vk4cJkZfG8vybST+erx7fC3C6yuxjgmGzH3qL4xxN3MZwOJDoy5GwHHM2vMdOwvvdm9p/xizOP60+pStSq7N3P/Uoyxyf1LNiD+4CSuRjDHT8jsFo3c+uuwzr0fB2HjN4/DaO+DP7gTMyUYV+3+Az+/t894FHbVvM5Dxrk/kwNYMnj7KX2cIhWJ4V66A/OuyznI4nDsLBBDCOywNcwq9MP87k7CdteOGQUeB7WuuavDywIgLreEKrO78yT4HvnjdZGytt5nGrShI5vk3nmGWBsIHx7sHhOnEbDI/OnAoWlzHGRCG2l82W0uuUaBRnep9zN0amw5zcdfRs+nNYirTrVzlKgUl83rjPExx1Tn2fuq/M2J5HK3rbQePxqr7yyry7f9ncfwnzrgrwSkQ6/VOunSAyNM5DVSoYdseaPc6VUvoN4RL5pZ+76UzzeylQgfuAam9jaB/Se7CRsFX6kzKqZUrCygZwuxU+hjFjSimfkwTAPMb+9RGmZMXjW+uI2k+BNJjR01l3/+/99N+KGLUTwr+4hXugOtNlLpeZApPHo+OxEaOXp+NB0BHh+lKlhlzEfz3nD8LQZVXZOl6ViE7XvbCifYPwL1aLk7dPG0y97BOGqT0R+6yyYtZrnyZns7XZMla3cl2/Fi2/TofmIZlnjIkr9rcP5OejO6dKEMU+Nsj73J1N3m4R3YzfleLY0q1aeH83WGV6KiNkZyrmFBR/qy1/G1QP2VkT+DhDjazNHoBzcSS+cMP8sn/dVfJxXFR0SfIFsp74sLlYmn6/3RKnah0V4uBr16uX+t09sG2KH5q3r9GIxCtd4GdrHXext4AjMofVlGHfSODtwW8PdLj4axnOAxsNk82JoReziLvWVegxfvbi4fr/RK09KN8jXYGU+9cM5UYZGNxyqprNBQgrDzMjWyPVYHMxQsl5st7MuL3YU/KxTbSpzoGSOGgQGxP9kyj4lH43bVWVyO/9fKRKIMSnKNOgZufKDxB1gOvrMDfWRdOfV35t6N87F9RPWQOTcIGWp5hi4c+QNazw/qMOvJRRVbl4TXZBLa+hpXhzAozy/N0Hs51Iloj8nO0i1Kxl2kYOUPUFsYaopnyli9PZs+tpGnK91fberDHNG+ff6wynx4DR1W7bPIDK+UW496NvPo6/hIamTyjHUNuLgDxtCjX/pq7ev8utigGuE6G2vZ8+P/4O/xqmHupixdhysqGvyRBu/07D9ROVk5fBaC0h+Uue37jUQIFLZgqmmodJh1lS7NEsD9JzKZJHTHqaV/LVZsVAPis7O7YsjvYKf+6E0d1Di8dgt6ptwfwBq7bdWZv/oGJP4EJuFoP/vPuLE++xIHtaqPLFXWmVery77MqNqvv/o9NLZpPfPMwl51dx5PsNmpGZHcSXxiN7WdsXKs9Jw9qZK/KDrdRODw/vWk++ImFL8re4OcZur6cRzwZw0sb8PWBPqsR8WAEBlWTBFMC4FydKxP+gusCMfk/KxFBddZDxcsMj7AaF/UHJejn8E6K3kBLPD3cRXjs+ktMrxwkPcQ5aXkXUx63yzkJpUR0uO5vcEfgbtX5pVQ0InPj2gxg9qaZrmGBG29nqdj/HE+8c9DTHNKpHrq4tWagwXF5nfeBf/84x9/+sf/A/jl+pU='))
assert hashlib.sha256(BOUNDARY_BYTES).hexdigest()==BOUNDARY_SHA256
REGION=shapely.geometry.shape(json.loads(BOUNDARY_BYTES)['features'][0]['geometry'])
assert REGION.is_valid
BBOX=(79.4,5.8,82.0,10.0)
x0,y0,x1,y1=REGION.bounds
assert BBOX[0]<=x0 and BBOX[1]<=y0 and BBOX[2]>=x1 and BBOX[3]>=y1
(OUT/'region.geojson').write_bytes(BOUNDARY_BYTES)
print('Same pilot polygon verified. Bounds:',REGION.bounds)


Same pilot polygon verified. Bounds: (79.65085150000002, 5.919017299999974, 81.87896019999998, 9.8355856)


## 3. Load comparison and year-planning helpers
These include the documented `N20` and `N21` codes. Unexpected platform codes remain errors. MODIS percentages and VIIRS confidence classes are never cross-mapped.


In [40]:
"""Compare native FIRMS standard-product API rows with the checked mirror extract.
Shared by local tests and the standalone notebook. Never logs or persists MAP_KEY.
"""
from io import BytesIO
from collections import Counter
import json
import numpy as np
import pandas as pd
import shapely

SOURCES={'MODIS_SP':{'T':'Terra','A':'Aqua','Terra':'Terra','Aqua':'Aqua'},
 'VIIRS_SNPP_SP':{'N':'Suomi-NPP','Suomi-NPP':'Suomi-NPP'},
 'VIIRS_NOAA20_SP':{'1':'NOAA-20','N20':'NOAA-20','NOAA-20':'NOAA-20'},
 'VIIRS_NOAA21_SP':{'N21':'NOAA-21','NOAA-21':'NOAA-21'}}


def parse_official_csv(raw,source,start,end,boundary):
    """Reject failed/error/changed-schema responses; clip a bounding-box result exactly."""
    if source not in SOURCES:raise ValueError('Unsupported standard-product source')
    try:frame=pd.read_csv(BytesIO(raw),dtype={'acq_time':'string','satellite':'string','confidence':'string','version':'string'})
    except Exception:raise ValueError('FIRMS response is not valid CSV; do not interpret it as zero') from None
    base={'latitude','longitude','acq_date','acq_time','satellite','instrument','confidence','version','type','frp','scan','track','daynight'}
    required=base|({'brightness','bright_t31'} if source=='MODIS_SP' else {'bright_ti4','bright_ti5'})
    if not required.issubset(frame):raise ValueError('FIRMS CSV schema is incomplete or changed; response is not an observed zero')
    # Whitespace is formatting only; platform aliases remain specific to each requested stream.
    codes=frame.satellite.astype('string').str.strip()
    satellite=codes.map(SOURCES[source])
    if satellite.isna().any():
        received={str(k):int(v) for k,v in codes.fillna('<missing>').value_counts(dropna=False).items()}
        unexpected=sorted(codes[satellite.isna()].fillna('<missing>').unique().tolist())
        raise ValueError(f'Unexpected satellite code for {source} [{start}, {end}): '
                         f'{json.dumps(unexpected)}. Received code counts: {json.dumps(received,sort_keys=True)}. '
                         'No rows were reclassified or discarded. Keep this source/date/code diagnostic for review.')
    times=frame.acq_time.astype('string').str.zfill(4)
    if not times.str.match(r'^\d{4}$').all():raise ValueError('Invalid acquisition time')
    try:
        acquired=pd.to_datetime(frame.acq_date.astype(str)+' '+times.str[:2]+':'+times.str[2:],utc=True,errors='raise')
        lon=pd.to_numeric(frame.longitude,errors='raise');lat=pd.to_numeric(frame.latitude,errors='raise')
    except Exception:raise ValueError('Invalid native date/time/coordinate fields') from None
    if not np.isfinite(lon).all() or not np.isfinite(lat).all() or not lon.between(-180,180).all() or not lat.between(-90,90).all():raise ValueError('Invalid coordinates')
    if not acquired.ge(pd.Timestamp(start,tz='UTC')).all() or not acquired.lt(pd.Timestamp(end,tz='UTC')).all():raise ValueError('API returned observations outside the requested window')
    frame['satellite']=satellite;frame['acq_datetime']=acquired;frame['audit_lon']=lon;frame['audit_lat']=lat
    if frame.version.str.contains('NRT',case=False,na=True).any():raise ValueError('NRT rows received for a standard-product query')
    if not frame.daynight.isin(['D','N']).all():raise ValueError('Invalid day/night value')
    if not frame.type.isin([0,1,2,3]).all():raise ValueError('Missing or unsupported standard-product type')
    if source=='MODIS_SP':
        numeric=pd.to_numeric(frame.confidence,errors='raise')
        if not (numeric.between(0,100)&numeric.eq(numeric.round())).all():raise ValueError('Invalid native MODIS confidence')
        frame['confidence']=numeric.map(lambda v:str(int(v)))
        frame['confidence_pct']=numeric
    else:
        if not frame.confidence.isin(['l','n','h']).all():raise ValueError('Invalid native VIIRS confidence')
        frame['confidence_pct']=np.nan
    for column in ['frp','scan','track']:
        numeric=pd.to_numeric(frame[column],errors='raise')
        if not np.isfinite(numeric).all() or numeric.lt(0).any():raise ValueError('Invalid native numeric field')
        frame[column]=numeric
    inside=shapely.covers(boundary,shapely.points(lon.to_numpy(),lat.to_numpy()))
    return frame.loc[inside].copy()


def native_keys(frame):
    """Published coordinates compared at 5 decimal degrees; multiplicities preserved."""
    times=pd.to_datetime(frame.acq_datetime,utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    return pd.Series([f'{s}|{t}|{float(x):.5f}|{float(y):.5f}' for s,t,x,y in zip(frame.satellite,times,frame.audit_lon,frame.audit_lat)],index=frame.index)


def compare_native(mirror,official):
    left=mirror.copy();right=official.copy();left['record_key']=native_keys(left);right['record_key']=native_keys(right)
    lc,rc=Counter(left.record_key),Counter(right.record_key)
    missing=[{'record_key':k,'rows':n} for k,n in (rc-lc).items()]
    extra=[{'record_key':k,'rows':n} for k,n in (lc-rc).items()]
    duplicate_keys=any(n>1 for n in lc.values()) or any(n>1 for n in rc.values())
    differences=[]
    if not duplicate_keys:
        l=left.set_index('record_key');r=right.set_index('record_key')
        columns=['confidence','type','daynight','instrument','version','frp','scan','track','brightness','bright_t31','bright_ti4','bright_ti5']
        common=l.index.intersection(r.index)
        numeric={'type','frp','scan','track','brightness','bright_t31','bright_ti4','bright_ti5'}
        for col in columns:
            if col not in l or col not in r:continue # sensor-inapplicable channels remain null / absent.
            a,b=l.loc[common,col],r.loc[common,col]
            if col in numeric:
                av=pd.to_numeric(a,errors='raise').to_numpy(dtype=float,na_value=np.nan);bv=pd.to_numeric(b,errors='raise').to_numpy(dtype=float,na_value=np.nan)
                same=np.isclose(av,bv,atol=1e-5,rtol=0,equal_nan=True)
            else:same=(a.astype('string').fillna('<NULL>')==b.astype('string').fillna('<NULL>')).to_numpy(dtype=bool)
            for key in common[~same]:differences.append({'record_key':key,'field':col,'mirror':str(l.loc[key,col]),'official':str(r.loc[key,col])})
    status='matched_native_records' if not (missing or extra or differences or duplicate_keys) else 'review_required'
    return {'status':status,'mirror_rows':len(left),'official_rows':len(right),'shared_rows':sum((lc&rc).values()),
      'official_rows_missing_in_mirror':sum((rc-lc).values()),'mirror_rows_not_in_official':sum((lc-rc).values()),
      'duplicate_keys_need_review':duplicate_keys,'native_attribute_differences':len(differences),
      'coordinate_comparison_decimals':5,'numeric_comparison_abs_tolerance':1e-5,
      'coverage_verified':False,'missing_in_mirror':missing,'extra_in_mirror':extra,'attribute_differences':differences}


In [41]:
"""Pure data helpers embedded in the standalone 2022/2024 Sri Lanka notebook."""
from html.parser import HTMLParser
from io import BytesIO
import json
import re
import pandas as pd
import numpy as np

YEAR_SATELLITES={2022:['Terra','Aqua','Suomi-NPP','NOAA-20'],2024:['Terra','Aqua','Suomi-NPP','NOAA-20','NOAA-21']}
PLATFORM_SOURCE={'Terra':'MODIS_SP','Aqua':'MODIS_SP','Suomi-NPP':'VIIRS_SNPP_SP','NOAA-20':'VIIRS_NOAA20_SP','NOAA-21':'VIIRS_NOAA21_SP'}
SOURCE_SENSOR={'MODIS_SP':'MODIS','VIIRS_SNPP_SP':'VIIRS_SNPP','VIIRS_NOAA20_SP':'VIIRS_NOAA20','VIIRS_NOAA21_SP':'VIIRS_NOAA21'}
PUBLIC_SOURCES={
 'snpp':'https://modaps.modaps.eosdis.nasa.gov/services/production/outages_suomi_npp.html',
 'terra':'https://modaps.modaps.eosdis.nasa.gov/services/production/outages_terra.html',
 'notices':'https://firms2.modaps.eosdis.nasa.gov/content/notifications/firms/outages.html',
 'missing':'https://firms.modaps.eosdis.nasa.gov/api/missing_data/?action=fetch_data'}

class EvidenceRows(HTMLParser):
    def __init__(self):super().__init__();self.rows=[];self.row=None;self.cell=None
    def handle_starttag(self,tag,attrs):
        if tag=='tr':self.row=[]
        elif tag in ['td','th']:self.cell=[]
        elif tag=='br' and self.cell is not None:self.cell.append(' ')
    def handle_data(self,text):
        if self.cell is not None:self.cell.append(text)
    def handle_endtag(self,tag):
        if tag in ['td','th'] and self.cell is not None:
            if self.row is not None:self.row.append(' '.join(''.join(self.cell).split()))
            self.cell=None
        elif tag=='tr' and self.row is not None:self.rows.append(self.row);self.row=None

def evidence_rows(text):
    p=EvidenceRows();p.feed(text);return p.rows

def availability_table(raw):
    try:table=pd.read_csv(BytesIO(raw),dtype=str)
    except Exception:raise ValueError('Availability lookup did not return CSV; no dates may be treated as zero') from None
    if not {'data_id','min_date','max_date'}.issubset(table):raise ValueError('NASA availability schema changed')
    table=table[table.data_id.isin(set(PLATFORM_SOURCE.values()))].copy()
    if table.data_id.duplicated().any():raise ValueError('Duplicate advertised source date ranges')
    for c in ['min_date','max_date']:
        if not table[c].str.fullmatch(r'\d{4}-\d{2}-\d{2}',na=False).all():raise ValueError('Invalid advertised date')
        pd.to_datetime(table[c],errors='raise')
    if (table.min_date>table.max_date).any():raise ValueError('Reversed advertised date range')
    return table

def plan_requests(available):
    plans=[];windows=[]
    by_source=available.set_index('data_id')
    for year,satellites in YEAR_SATELLITES.items():
        for source in dict.fromkeys(PLATFORM_SOURCE[s] for s in satellites):
            begin=pd.Timestamp(year,1,1);stop=pd.Timestamp(year+1,1,1)
            if source not in by_source.index:
                plans.append({'year':year,'source':source,'status':'standard_source_not_advertised','start':None,'end':None,'days':0});continue
            a=max(begin,pd.Timestamp(by_source.loc[source,'min_date']))
            b=min(stop,pd.Timestamp(by_source.loc[source,'max_date'])+pd.Timedelta(days=1))
            if a>=b:
                plans.append({'year':year,'source':source,'status':'outside_advertised_archive_range','start':None,'end':None,'days':0});continue
            status='full_year_advertised' if a==begin and b==stop else 'partial_year_advertised'
            plans.append({'year':year,'source':source,'status':status,'start':str(a.date()),'end':str(b.date()),'days':(b-a).days})
            start=a
            while start<b:
                end=min(start+pd.Timedelta(days=5),b)
                windows.append((year,source,str(start.date()),str(end.date())));start=end
    plans=pd.DataFrame(plans)
    for column in ['start','end']:plans[column]=plans[column].astype(object).where(plans[column].notna(),None)
    return plans,windows

def stack_native(parts):
    columns=list(dict.fromkeys(c for p in parts for c in p.columns))
    nonempty=[p for p in parts if len(p)]
    return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()

def parse_gap_evidence(public):
    intervals=[]
    for label,satellite in [('snpp','Suomi-NPP'),('terra','Terra')]:
        for row in evidence_rows(public[label].decode()):
            if not row or not re.match(r'^(2022|2024)-\d{3}(?:\s*-\s*(?:2022|2024)-\d{3})?$',row[0]):continue
            if len(row)!=5:raise ValueError('Production outage table schema changed')
            dates=re.findall(r'20\d\d-\d{3}',row[0]);a=pd.to_datetime(dates[0],format='%Y-%j',utc=True)+pd.Timedelta(row[2]);b=pd.to_datetime(dates[-1],format='%Y-%j',utc=True)+pd.Timedelta(row[3])
            if b<=a:raise ValueError('Ambiguous outage end time; do not guess an interval')
            intervals.append({'satellite':satellite,'start_utc':a,'end_utc':b,'source_url':PUBLIC_SOURCES[label],
                'source_year_day':row[0],'source_reason':row[4],'local_impact':'unverified','recovery_status':'not_stated_in_table',
                'interval_semantics':'reported production/granule window; endpoints retained as published'})
    payload=json.loads(public['missing'])
    if payload.get('success') is not True or not isinstance(payload.get('html'),str):raise ValueError('Public missing-date lookup failed; no empty inventory inferred')
    platform={'Terra (MODIS)':'Terra','Aqua (MODIS)':'Aqua','Suomi NPP (VIIRS)':'Suomi-NPP','NOAA-20 (VIIRS)':'NOAA-20','NOAA-21 (VIIRS)':'NOAA-21'}
    missing=[]
    for row in evidence_rows(payload['html']):
        if len(row)!=4:raise ValueError('Missing-date table schema changed')
        if row[0].startswith(('2022-','2024-')):
            if row[1] not in platform:raise ValueError('Unknown missing-date platform')
            pd.Timestamp(row[0]);missing.append({'date':row[0],'satellite':platform[row[1]],'quality':'sp','source_url':PUBLIC_SOURCES['missing']})
    return pd.DataFrame(intervals,columns=['satellite','start_utc','end_utc','source_url','source_year_day','source_reason','local_impact','recovery_status','interval_semantics']),pd.DataFrame(missing,columns=['date','satellite','quality','source_url'])

def diagnostic_grids(mirrors,officials,plans,intervals,missing):
    daily=[];grids=[]
    gaps=set(zip(missing.date,missing.satellite))
    for year,satellites in YEAR_SATELLITES.items():
        m=mirrors[year].copy();m['date']=pd.to_datetime(m.acq_datetime,utc=True).dt.strftime('%Y-%m-%d');m['month']=m.date.str[:7]
        if not m.quality.isin(['sp','nrt']).all():raise ValueError('Unknown mirror quality')
        if not m.loc[m.quality.eq('sp'),'type'].isin([0,1,2,3]).all():raise ValueError('Unknown standard type domain')
        if not m.satellite.isin(satellites).all():raise ValueError('Unexpected platform in the yearly mirror contract')
        official=officials[year].copy();official['date']=pd.to_datetime(official.acq_datetime,utc=True).dt.strftime('%Y-%m-%d')
        for satellite in satellites:
            days=pd.date_range(f'{year}-01-01',f'{year}-12-31').strftime('%Y-%m-%d')
            frame=pd.DataFrame({'year':year,'date':days,'satellite':satellite})
            local=m[m.satellite.eq(satellite)]
            for quality in ['sp','nrt']:
                counts=local[local.quality.eq(quality)].groupby('date').size()
                frame['mirror_'+quality+'_detections']=frame.date.map(counts).fillna(0).astype(int)
            frame['official_sp_detections']=pd.Series(pd.NA,index=frame.index,dtype='Int64')
            plan=plans[plans.year.eq(year)&plans.source.eq(PLATFORM_SOURCE[satellite])].iloc[0]
            frame['official_query_state']=plan.status
            queried=frame.date.ge(plan.start)&frame.date.lt(plan.end) if plan.days else pd.Series(False,index=frame.index)
            counts=official[official.satellite.eq(satellite)].groupby('date').size()
            frame.loc[queried,'official_sp_detections']=frame.loc[queried,'date'].map(counts).fillna(0).astype('Int64')
            frame.loc[queried,'official_query_state']='query_completed'
            frame.loc[~queried&bool(plan.days),'official_query_state']='outside_advertised_archive_range'
            frame['upstream_lists_missing_standard_data']=[(d,satellite) in gaps for d in frame.date]
            frame['documented_global_gap_on_date']=False
            for event in intervals[intervals.satellite.eq(satellite)].itertuples():
                overlap=pd.date_range(event.start_utc.floor('D'),(event.end_utc-pd.Timedelta(nanoseconds=1)).floor('D')).strftime('%Y-%m-%d')
                frame.loc[frame.date.isin(overlap),'documented_global_gap_on_date']=True
            frame['observing_coverage']='unverified';frame['coverage_adjusted_activity']=np.nan;daily.append(frame)
            domain=[str(i) for i in range(101)] if satellite in ['Terra','Aqua'] else ['l','n','h']
            sp=local[local.quality.eq('sp')].copy();sp['native_confidence']=sp.confidence.astype(str)
            if not sp.native_confidence.isin(domain).all():raise ValueError('Native confidence changed; no crosswalk applied')
            idx=pd.MultiIndex.from_product([[f'{year}-{i:02}' for i in range(1,13)],[satellite],[0,1,2,3],domain],names=['month','satellite','type','native_confidence'])
            counts=sp.groupby(idx.names).size().reindex(idx,fill_value=0).rename('recorded_detections').reset_index()
            counts['quality']='sp';counts['confidence_scale']='MODIS_integer' if satellite in ['Terra','Aqua'] else 'VIIRS_class';counts['observing_coverage']='unverified'
            counts['record_state']=np.where(counts.recorded_detections.gt(0),'recorded_positive','recorded_zero_query_completed');grids.append(counts)
    daily=pd.concat(daily,ignore_index=True);daily['month']=daily.date.str[:7]
    monthly=daily.groupby(['year','month','satellite'],sort=False).agg(mirror_sp_detections=('mirror_sp_detections','sum'),mirror_nrt_detections=('mirror_nrt_detections','sum'),
        official_sp_on_queried_days=('official_sp_detections',lambda x:x.sum(min_count=1)),days_in_month=('date','size'),days_queried=('official_query_state',lambda x:int(x.eq('query_completed').sum())),
        upstream_listed_missing_days=('upstream_lists_missing_standard_data','sum'),dates_overlapping_global_gap=('documented_global_gap_on_date','sum')).reset_index()
    monthly['official_full_month_sp_detections']=monthly.official_sp_on_queried_days.where(monthly.days_queried.eq(monthly.days_in_month)).astype('Int64')
    monthly['observing_coverage']='unverified'
    return daily,monthly,pd.concat(grids,ignore_index=True)


## 4. Private key and safe requests
No credential-bearing URL is printed or saved. Quota/network errors retain completed files; rerun after resolving the error.


In [42]:
MAP_KEY=getpass('NASA FIRMS MAP_KEY (hidden): ').strip()
assert MAP_KEY and MAP_KEY.replace('-','').isalnum(),'Enter a FIRMS MAP_KEY, not an Earthdata password.'
def fetch_bytes(url,private=False):
    last='network/timeout'
    for attempt in range(5):
        try:
            with urlopen(Request(url,headers={'User-Agent':'CinderTrace-data-checkpoint/2.0','X-Requested-With':'XMLHttpRequest'}),timeout=90) as response:
                payload=response.read()
            if private and MAP_KEY.encode() in payload:
                raise ValueError('Provider echoed a credential; response was not saved.')
            return payload
        except HTTPError as error:
            last='HTTP '+str(error.code)
            if error.code not in [429,500,502,503,504]:break
        except (URLError,TimeoutError,OSError):last='network/timeout'
        if attempt<4:time.sleep(min(2**(attempt+1),30))
    raise RuntimeError(f'Public data request failed ({last}). No zero was written. Check service/key/quota and rerun; verified progress is cached.') from None

def save_receipt(path,payload,request):
    receipt={'request':request,'retrieved_utc':datetime.now(timezone.utc).isoformat(),
        'bytes':len(payload),'sha256':hashlib.sha256(payload).hexdigest()}
    temporary=path.with_suffix(path.suffix+'.part');temporary.write_bytes(payload);temporary.replace(path)
    receipt_path=Path(str(path)+'.receipt.json');receipt_path.write_text(json.dumps(receipt,indent=2))
    return receipt

def cached_bytes(path,url,request,private=False):
    receipt_path=Path(str(path)+'.receipt.json')
    if path.exists() and receipt_path.exists():
        raw=path.read_bytes();receipt=json.loads(receipt_path.read_text())
        assert receipt['request']==request and receipt['bytes']==len(raw) and receipt['sha256']==hashlib.sha256(raw).hexdigest(), 'Cached input differs; preserve it and start a new run for review.'
        return raw,receipt
    raw=fetch_bytes(url,private=private)
    return raw,save_receipt(path,raw,request)
print('Private key entered. Request helpers ready.')


Private key entered. Request helpers ready.


## 5. Pin NASA archive availability and outage evidence
Archive date availability is not cloud/overpass completeness. Source notices are retained as evidence. The inventory covers the retrieved S-NPP/Terra production tables and FIRMS missing standard dates, not every possible satellite issue. Recovery announcements are preserved in their original notice page; no exact recovery time is guessed.


In [43]:
availability_raw,availability_receipt=cached_bytes(OUT/'nasa_availability.csv',
    'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/'+MAP_KEY+'/all',
    {'service':'NASA FIRMS data_availability','source':'all'},private=True)
AVAILABLE=availability_table(availability_raw)
PLANS,WINDOWS=plan_requests(AVAILABLE)
PLANS.to_parquet(OUT/'official_request_plan.parquet',index=False)
display(PLANS)
assert all(0<(pd.Timestamp(b)-pd.Timestamp(a)).days<=5 for _,_,a,b in WINDOWS)
print('Planned NASA requests:',len(WINDOWS))
PUBLIC={};PUBLIC_RECEIPTS={}
for label,url in PUBLIC_SOURCES.items():
    raw,receipt=cached_bytes(OUT/('source_'+label+('.json' if label=='missing' else '.html')),url,{'public_source':url})
    PUBLIC[label]=raw;PUBLIC_RECEIPTS[label]=receipt
GAPS,MISSING_DATES=parse_gap_evidence(PUBLIC)
GAPS.to_parquet(OUT/'documented_global_gaps.parquet',index=False)
MISSING_DATES.to_parquet(OUT/'upstream_missing_standard_dates.parquet',index=False)
print('Global production-gap intervals:',len(GAPS),'| listed standard missing dates:',len(MISSING_DATES))
print('Unlisted dates remain unverified; a global gap does not establish the exact lost local passes.')


,year,source,status,start,end,days
0,2022,MODIS_SP,full_year_advertised,2022-01-01,2023-01-01,365
1,2022,VIIRS_SNPP_SP,full_year_advertised,2022-01-01,2023-01-01,365
2,2022,VIIRS_NOAA20_SP,full_year_advertised,2022-01-01,2023-01-01,365
3,2024,MODIS_SP,full_year_advertised,2024-01-01,2025-01-01,366
4,2024,VIIRS_SNPP_SP,full_year_advertised,2024-01-01,2025-01-01,366
5,2024,VIIRS_NOAA20_SP,full_year_advertised,2024-01-01,2025-01-01,366
6,2024,VIIRS_NOAA21_SP,full_year_advertised,2024-01-01,2025-01-01,366


Planned NASA requests: 515
Global production-gap intervals: 122 | listed standard missing dates: 63
Unlisted dates remain unverified; a global gap does not establish the exact lost local passes.


## 6. Extract the two mirror years
Each extraction records schema, SQL, parameters, before/after source fingerprints and an independent direct count. Complete raw fields are retained, including any NRT rows. Only successful queries can emit a recorded zero. Source ETag is an object fingerprint, not a checksum. Cached extracts are checked locally; start a new run to examine a revised source snapshot.


In [44]:
con=duckdb.connect()
con.execute("SET memory_limit='2GB'");con.execute('SET threads=2')
con.execute('INSTALL httpfs');con.execute('LOAD httpfs');con.execute('INSTALL spatial');con.execute('LOAD spatial')
WHERE_SQL="""acq_date >= CAST(? AS DATE) AND acq_date < CAST(? AS DATE)
AND ST_Intersects(geometry,ST_MakeEnvelope(?,?,?,?)) AND ST_Covers(ST_GeomFromText(?),geometry)"""
EXTRACT_SQL=f"SELECT * EXCLUDE(geometry),ST_AsWKB(geometry) AS geometry_wkb,ST_X(geometry) AS audit_lon,ST_Y(geometry) AS audit_lat FROM read_parquet(?) WHERE {WHERE_SQL}"

def fingerprint(url):
    try:
        with urlopen(Request(url,method='HEAD',headers={'User-Agent':'CinderTrace-data-checkpoint/2.0'}),timeout=60) as response:
            result={k:response.headers.get(k) for k in ['ETag','Content-Length','Last-Modified']}
    except Exception:raise RuntimeError('Mirror object fingerprint request failed; rerun without inventing a snapshot.') from None
    assert result['ETag'] and result['Content-Length'],'Missing object fingerprint; stop for review.'
    return result

def validate_mirror(frame,year):
    required={'acq_datetime','acq_date','satellite','sensor','instrument','quality','confidence','confidence_pct','type','version','frp','daynight','scan','track','geometry_wkb','audit_lon','audit_lat'}
    assert required<=set(frame.columns),'Mirror schema changed.'
    times=pd.to_datetime(frame.acq_datetime,utc=True)
    assert times.dt.year.eq(year).all() and times.dt.date.eq(pd.to_datetime(frame.acq_date).dt.date).all()
    if 'year' in frame:assert frame.year.eq(year).all(),'Partition year differs from acquisition year.'
    assert frame.satellite.isin(YEAR_SATELLITES[year]).all() and frame.quality.isin(['sp','nrt']).all()
    assert frame.sensor.eq(frame.satellite.map({s:SOURCE_SENSOR[PLATFORM_SOURCE[s]] for s in YEAR_SATELLITES[year]})).all()
    geometry=shapely.from_wkb(frame.geometry_wkb.map(bytes).to_numpy())
    assert shapely.is_valid(geometry).all() and shapely.covers(REGION,geometry).all()
    assert np.allclose(shapely.get_x(geometry),frame.audit_lon,atol=1e-9,rtol=0) and np.allclose(shapely.get_y(geometry),frame.audit_lat,atol=1e-9,rtol=0)
    assert not frame.duplicated().any(),'Exact duplicate rows; retain the extract for investigation.'
    modis=frame.sensor.eq('MODIS')
    assert frame.loc[modis,'confidence_pct'].between(0,100).all()
    assert pd.to_numeric(frame.loc[modis,'confidence']).eq(frame.loc[modis,'confidence_pct']).all()
    assert frame.loc[~modis,'confidence'].isin(['l','n','h']).all() and frame.loc[~modis,'confidence_pct'].isna().all()
    assert frame.daynight.isin(['D','N']).all() and frame.loc[frame.quality.eq('sp'),'type'].isin([0,1,2,3]).all()
    assert np.isfinite(frame.frp).all() and frame.frp.ge(0).all()
    return frame

MIRRORS={};MIRROR_RECEIPTS={}
try:
    for year in YEAR_SATELLITES:
        path=OUT/f'mirror_{year}.parquet';receipt_path=OUT/f'mirror_{year}.receipt.json'
        if path.exists() and receipt_path.exists():
            receipt=json.loads(receipt_path.read_text());raw=path.read_bytes()
            assert receipt['year']==year and receipt['boundary_sha256']==BOUNDARY_SHA256
            assert receipt['sha256']==hashlib.sha256(raw).hexdigest() and receipt['bytes']==len(raw)
            frame=validate_mirror(pd.read_parquet(BytesIO(raw)),year)
            assert len(frame)==receipt['direct_count']==receipt['rows'] and receipt['source_before']==receipt['source_after']
        else:
            url=f'https://data.source.coop/portolan-mirrors/firms-catalog/detections/year={year}/detections.parquet'
            before=fingerprint(url)
            schema=con.execute('DESCRIBE SELECT * FROM read_parquet(?)',[url]).df()
            assert str(schema.set_index('column_name').loc['geometry','column_type']).startswith('GEOMETRY'),'Native geometry decoding failed.'
            assert not {'geometry_wkb','audit_lon','audit_lat'}&set(schema.column_name),'Audit column collision.'
            params=[url,f'{year}-01-01',f'{year+1}-01-01',*BBOX,REGION.wkt]
            frame=con.execute(EXTRACT_SQL,params).df();frame['geometry_wkb']=frame.geometry_wkb.map(bytes)
            direct=int(con.execute(f'SELECT count(*) FROM read_parquet(?) WHERE {WHERE_SQL}',params).fetchone()[0])
            after=fingerprint(url)
            assert before==after and len(frame)==direct,'Source/count changed; preserve this run and retry in a new run folder.'
            frame=validate_mirror(frame,year)
            temporary=path.with_suffix('.part.parquet');frame.to_parquet(temporary,index=False);temporary.replace(path)
            raw=path.read_bytes()
            receipt={'year':year,'source_url':url,'source_before':before,'source_after':after,'etag_is_not_checksum':True,
                'boundary_sha256':BOUNDARY_SHA256,'sql':EXTRACT_SQL,'parameters':params,'rows':len(frame),'direct_count':direct,
                'sha256':hashlib.sha256(raw).hexdigest(),'bytes':len(raw),'schema':schema.to_dict('records'),'retrieved_utc':datetime.now(timezone.utc).isoformat()}
            receipt_path.write_text(json.dumps(receipt,indent=2,default=str))
        MIRRORS[year]=frame;MIRROR_RECEIPTS[str(year)]=receipt
        print(year,'checked mirror rows:',len(frame));display(frame.groupby(['satellite','quality'],dropna=False).size().rename('recorded_detections').reset_index())
finally:con.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2022 checked mirror rows: 8593


,satellite,quality,recorded_detections
0,Aqua,sp,395
1,NOAA-20,sp,3917
2,Suomi-NPP,sp,4079
3,Terra,sp,202


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2024 checked mirror rows: 17157


,satellite,quality,recorded_detections
0,Aqua,sp,419
1,NOAA-20,sp,5813
2,NOAA-21,nrt,5775
3,Suomi-NPP,sp,5010
4,Terra,sp,140


## 7. Retrieve NASA standard observations
Files are cached per source and half-open UTC interval. Every response is parsed and clipped before its regional count is accepted. Previously cached bytes are reparsed on resume. Missing archive dates and NRT rows are not forced into the standard-product comparison.


In [46]:
OFFICIAL_PARTS={year:[] for year in YEAR_SATELLITES};REQUEST_RECEIPTS=[]
for index,(year,source,start,end) in enumerate(WINDOWS,1):
    request={'year':year,'source':source,'start_inclusive_utc':start,'end_exclusive_utc':end,'bbox':list(BBOX),'boundary_sha256':BOUNDARY_SHA256}
    stem=f'{source}_{start}_{end}';path=CACHE/(stem+'.csv')
    url='https://firms.modaps.eosdis.nasa.gov/api/area/csv/'+MAP_KEY+'/'+source+'/'+','.join(map(str,BBOX))+'/'+str((pd.Timestamp(end)-pd.Timestamp(start)).days)+'/'+start
    payload,receipt=cached_bytes(path,url,request,private=True)
    try:frame=parse_official_csv(payload,source,start,end,REGION)
    except ValueError as error:
        raise ValueError(f'Unverified NASA response for {source} {start} to {end}: {error}. Raw response retained but not counted; stop for review.') from None
    REQUEST_RECEIPTS.append({**receipt,'regional_rows':len(frame),'request_status':'csv_schema_and_scope_verified'})
    OFFICIAL_PARTS[year].append(frame)
    if index%15==0 or index==len(WINDOWS):print(f'Checked {index}/{len(WINDOWS)} requests; {sum(len(f) for parts in OFFICIAL_PARTS.values() for f in parts):,} regional standard observations.')
    time.sleep(0.25)
OFFICIALS={}
for year,parts in OFFICIAL_PARTS.items():
    assert parts,'No advertised standard sources for this year; review the availability response.'
    OFFICIALS[year]=stack_native(parts)
    OFFICIALS[year].to_parquet(OUT/f'official_{year}.parquet',index=False)
assert len(REQUEST_RECEIPTS)==len(WINDOWS)
print('All planned requests completed. Unsupported dates remain unqueried, not zero.')


Checked 15/515 requests; 84 regional standard observations.
Checked 30/515 requests; 146 regional standard observations.
Checked 45/515 requests; 389 regional standard observations.
Checked 60/515 requests; 588 regional standard observations.
Checked 75/515 requests; 603 regional standard observations.
Checked 90/515 requests; 1,131 regional standard observations.
Checked 105/515 requests; 1,482 regional standard observations.
Checked 120/515 requests; 2,687 regional standard observations.
Checked 135/515 requests; 4,627 regional standard observations.
Checked 150/515 requests; 4,731 regional standard observations.
Checked 165/515 requests; 5,474 regional standard observations.
Checked 180/515 requests; 5,785 regional standard observations.
Checked 195/515 requests; 6,999 regional standard observations.
Checked 210/515 requests; 8,547 regional standard observations.
Checked 225/515 requests; 8,596 regional standard observations.
Checked 240/515 requests; 8,714 regional standard observa

/tmp/ipykernel_476/2403494387.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


All planned requests completed. Unsupported dates remain unqueried, not zero.


/tmp/ipykernel_476/2403494387.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


## 8. Reconcile native rows and retain unsupported observations
Comparison covers only standard mirror rows within the advertised NASA intervals. Every excluded mirror row is saved with a reason, including all NRT rows. A match on the supported interval does not certify an unsupported remainder. Different satellites are never substitutes for one another.


In [47]:
COMPARISONS={};EXCLUDED=[];source_results=[]
for year,mirror in MIRRORS.items():
    covered=pd.Series(False,index=mirror.index);times=pd.to_datetime(mirror.acq_datetime,utc=True)
    for plan in PLANS[PLANS.year.eq(year)].itertuples():
        sats=[s for s in YEAR_SATELLITES[year] if PLATFORM_SOURCE[s]==plan.source]
        mask=mirror.quality.eq('sp')&mirror.satellite.isin(sats)
        if plan.days:
            mask&=times.ge(pd.Timestamp(plan.start,tz='UTC'))&times.lt(pd.Timestamp(plan.end,tz='UTC'));covered|=mask
            official=OFFICIALS[year][OFFICIALS[year].satellite.isin(sats)]
            result=compare_native(mirror.loc[mask],official)
            source_results.append({'year':year,'source':plan.source,'archive_scope':plan.status,**{k:v for k,v in result.items() if k not in ['missing_in_mirror','extra_in_mirror','attribute_differences']}})
        else:source_results.append({'year':year,'source':plan.source,'archive_scope':plan.status,'status':'not_checked_archive_unavailable'})
    excluded=mirror.loc[~covered].copy()
    excluded['not_reconciled_reason']=np.where(excluded.quality.ne('sp'),'NRT_retained_not_compared_as_standard','outside_advertised_standard_archive_range')
    excluded.to_parquet(OUT/f'mirror_not_reconciled_{year}.parquet',index=False);EXCLUDED.append(excluded)
    result=compare_native(mirror.loc[covered],OFFICIALS[year])
    COMPARISONS[str(year)]={**result,'scope_completeness':'full_declared_year' if PLANS[PLANS.year.eq(year)].status.eq('full_year_advertised').all() else 'partial_by_advertised_archive_dates',
        'mirror_rows_not_reconciled':len(excluded),'mirror_nrt_rows':int(mirror.quality.eq('nrt').sum()),'scientific_coverage_verified':False}
    for key in ['missing_in_mirror','extra_in_mirror','attribute_differences']:
        pd.DataFrame(result[key]).to_parquet(OUT/f'{key}_{year}.parquet',index=False)
SOURCE_RESULTS=pd.DataFrame(source_results);SOURCE_RESULTS.to_parquet(OUT/'source_reconciliation.parquet',index=False)
display(SOURCE_RESULTS)
print(json.dumps({year:{k:v for k,v in report.items() if k not in ['missing_in_mirror','extra_in_mirror','attribute_differences']} for year,report in COMPARISONS.items()},indent=2))


,year,source,archive_scope,status,mirror_rows,official_rows,shared_rows,official_rows_missing_in_mirror,mirror_rows_not_in_official,duplicate_keys_need_review,native_attribute_differences,coordinate_comparison_decimals,numeric_comparison_abs_tolerance,coverage_verified
0,2022,MODIS_SP,full_year_advertised,matched_native_records,597,597,597,0,0,False,0,5,0.00001,False
1,2022,VIIRS_SNPP_SP,full_year_advertised,matched_native_records,4079,4079,4079,0,0,False,0,5,0.00001,False
2,2022,VIIRS_NOAA20_SP,full_year_advertised,matched_native_records,3917,3917,3917,0,0,False,0,5,0.00001,False
3,2024,MODIS_SP,full_year_advertised,matched_native_records,559,559,559,0,0,False,0,5,0.00001,False
4,2024,VIIRS_SNPP_SP,full_year_advertised,matched_native_records,5010,5010,5010,0,0,False,0,5,0.00001,False
5,2024,VIIRS_NOAA20_SP,full_year_advertised,matched_native_records,5813,5813,5813,0,0,False,0,5,0.00001,False
6,2024,VIIRS_NOAA21_SP,full_year_advertised,review_required,0,5654,0,5654,0,False,0,5,0.00001,False


{
  "2022": {
    "status": "matched_native_records",
    "mirror_rows": 8593,
    "official_rows": 8593,
    "shared_rows": 8593,
    "official_rows_missing_in_mirror": 0,
    "mirror_rows_not_in_official": 0,
    "duplicate_keys_need_review": false,
    "native_attribute_differences": 0,
    "coordinate_comparison_decimals": 5,
    "numeric_comparison_abs_tolerance": 1e-05,
    "coverage_verified": false,
    "scope_completeness": "full_declared_year",
    "mirror_rows_not_reconciled": 0,
    "mirror_nrt_rows": 0,
    "scientific_coverage_verified": false
  },
  "2024": {
    "status": "review_required",
    "mirror_rows": 11382,
    "official_rows": 17036,
    "shared_rows": 11382,
    "official_rows_missing_in_mirror": 5654,
    "mirror_rows_not_in_official": 0,
    "duplicate_keys_need_review": false,
    "native_attribute_differences": 0,
    "coordinate_comparison_decimals": 5,
    "numeric_comparison_abs_tolerance": 1e-05,
    "coverage_verified": false,
    "scope_completeness

## 9. Daily, monthly and native filter diagnostics
The standard confidence grid is complete within each year/platform scope. NRT rows are preserved separately in the raw extract and unreconciled tables. Null authoritative counts mean dates were not queried; recorded zero requires a completed query. Outage flags are global evidence, not lost-fire estimates. No reference-equivalent seasons, transfer models or uncertainty bands are computed.


In [48]:
DAILY,MONTHLY,NATIVE_GRID=diagnostic_grids(MIRRORS,OFFICIALS,PLANS,GAPS,MISSING_DATES)
assert len(DAILY)==3290 and len(MONTHLY)==108 and len(NATIVE_GRID)==20112
assert len(DAILY[DAILY.date.eq('2024-02-29')])==5
assert NATIVE_GRID.recorded_detections.sum()==sum(int(m.quality.eq('sp').sum()) for m in MIRRORS.values())
DAILY.to_parquet(OUT/'daily_diagnostics.parquet',index=False)
MONTHLY.to_parquet(OUT/'monthly_diagnostics.parquet',index=False)
NATIVE_GRID.to_parquet(OUT/'native_standard_filter_grid.parquet',index=False)
conflicts=DAILY[DAILY.upstream_lists_missing_standard_data&DAILY.official_sp_detections.fillna(0).gt(0)]
conflicts.to_parquet(OUT/'missing_date_positive_record_review.parquet',index=False)
controls=[]
for event in GAPS.itertuples():
    for year,mirror in MIRRORS.items():
        times=pd.to_datetime(mirror.acq_datetime,utc=True)
        if event.start_utc.year!=year:continue
        controls.append({'year':year,'satellite':event.satellite,'start_utc':event.start_utc,'end_utc':event.end_utc,
            'mirror_standard_records_in_reported_window':int((mirror.satellite.eq(event.satellite)&mirror.quality.eq('sp')&times.ge(event.start_utc)&times.lt(event.end_utc)).sum()),
            'source_url':event.source_url,'interpretation':'global reported interval; local impact/recovery not established; no rows deleted'})
pd.DataFrame(controls).to_parquet(OUT/'outage_controls.parquet',index=False)
display(MONTHLY)
print('All raw/derived counts reconcile. Upstream missing-date entries with positive local standard rows needing review:',len(conflicts))
print('Recorded zeros remain recorded zeros; observing coverage and harmonization remain unverified.')


,year,month,satellite,mirror_sp_detections,mirror_nrt_detections,official_sp_on_queried_days,days_in_month,days_queried,upstream_listed_missing_days,dates_overlapping_global_gap,official_full_month_sp_detections,observing_coverage
0,2022,2022-01,Terra,5,0,5,31,31,0,1,5,unverified
1,2022,2022-02,Terra,8,0,8,28,28,0,1,8,unverified
2,2022,2022-03,Terra,9,0,9,31,31,0,0,9,unverified
3,2022,2022-04,Terra,9,0,9,30,30,0,1,9,unverified
4,2022,2022-05,Terra,11,0,11,31,31,0,0,11,unverified
...,...,...,...,...,...,...,...,...,...,...,...,...
103,2024,2024-08,NOAA-21,0,610,574,31,31,0,0,574,unverified
104,2024,2024-09,NOAA-21,0,2410,2285,30,30,0,0,2285,unverified
105,2024,2024-10,NOAA-21,0,128,126,31,31,0,0,126,unverified
106,2024,2024-11,NOAA-21,0,15,14,30,30,0,0,14,unverified


All raw/derived counts reconcile. Upstream missing-date entries with positive local standard rows needing review: 0
Recorded zeros remain recorded zeros; observing coverage and harmonization remain unverified.


## 10. Save the evidence bundle
The manifest hashes every included file, including raw NASA responses, availability/notice snapshots and mirror extracts. API credentials are excluded. Return this ZIP and the executed notebook; do not update the frontend or fit transfer models yet.


In [49]:
manifest={'run_id':RUN_ID,'years':[2022,2024],'scope':'Sri Lanka original pilot polygon; native observations, satellites/quality retained',
    'year_satellites':YEAR_SATELLITES,'boundary_sha256':BOUNDARY_SHA256,'bbox':list(BBOX),
    'preceding_2023_checkpoint_sha256':'509afdf0418a01b248671addb4d7a221a02e0e5e275fb203aa6e170206d29077',
    'availability_receipt':availability_receipt,'public_source_receipts':PUBLIC_RECEIPTS,'mirror_receipts':MIRROR_RECEIPTS,
    'official_request_plan':PLANS.to_dict('records'),'expected_requests':len(WINDOWS),'completed_requests':len(REQUEST_RECEIPTS),
    'request_receipts':REQUEST_RECEIPTS,'comparisons':COMPARISONS,
    'all_source_dates_advertised_full_year':bool(PLANS.status.eq('full_year_advertised').all()),
    'coverage_verified':False,'harmonization_applied':False,'production_export_enabled':False,'frontend_changed':False,
    'packages':{p:metadata.version(p) for p in ['duckdb','pandas','pyarrow','numpy','shapely']},
    'completed_utc':datetime.now(timezone.utc).isoformat(),'files':{}}
for path in sorted(OUT.rglob('*')):
    if not path.is_file() or path.name=='checkpoint_manifest.json':continue
    payload=path.read_bytes()
    assert MAP_KEY.encode() not in payload,'Credential found in output; stop before bundling.'
    manifest['files'][str(path.relative_to(OUT))]={'bytes':len(payload),'sha256':hashlib.sha256(payload).hexdigest()}
manifest_bytes=json.dumps(manifest,indent=2,default=str,allow_nan=False).encode()
assert MAP_KEY.encode() not in manifest_bytes
(OUT/'checkpoint_manifest.json').write_bytes(manifest_bytes)
ZIP_PATH=BASE/f'cindertrace_atlas_lk_2022_2024_{RUN_ID}.zip'
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for name in sorted([*manifest['files'],'checkpoint_manifest.json']):archive.write(OUT/name,name)
print('Checkpoint:',ZIP_PATH)
print('SHA256:',hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Save this executed notebook unchanged with the ZIP. A partial archive scope or mismatch remains explicit in the manifest.')
MAP_KEY=None
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))


Checkpoint: /content/drive/MyDrive/cindertrace_atlas/verification/lk_2022_2024/cindertrace_atlas_lk_2022_2024_20261114T201123Z_30cc8ecb.zip
SHA256: 6203a50c88921de8c29757192fbcbd9ee2af3ee06bd7f923e5dcf50cc34deea5
Save this executed notebook unchanged with the ZIP. A partial archive scope or mismatch remains explicit in the manifest.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>